In [6]:
# %pip install langchain-ollama

In [ ]:
# %pip install -U langgraph langchain-tavily langgraph-checkpoint-sqlite

In [18]:
# %pip install langchain

In [1]:
# %pip install -U langchain-community

In [2]:
from langchain_ollama.llms import OllamaLLM

llm = OllamaLLM(model="gemma3:270m")

In [8]:
from langchain.prompts import PromptTemplate

prompt = PromptTemplate.from_template("Give {number} names for a {domain} startup?")

In [9]:
chain = prompt | llm

In [11]:
response = chain.invoke({'number': 3, 'domain': 'tech'})
print(response)

Here are 3 tech startup names:

1.  **Synapse AI** - Evokes the idea of connecting minds and processing information.
2.  **Verity Labs** - Suggests a reliable, trustworthy, and accurate solution.
3.  **Momentum Solutions** - Implies progress, innovation, and a focus on achieving goals.


Agent

In [12]:
from langchain.agents import tool
import datetime

@tool
def get_current_datetime(format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """
    Returns the current date and time, formatted according to the provided Python strftime format string.
    Use this tool whenever the user asks for the current date, time, or both.
    Example format strings: '%Y-%m-%d' for date, '%H:%M:%S' for time.
    If no format is specified, defaults to '%Y-%m-%d %H:%M:%S'.
    """
    try:
        return datetime.datetime.now().strftime(format)
    except Exception as e:
        return f"Error formatting date/time: {e}"

# Put all your tools in a list
tools = [get_current_datetime]

In [13]:
from langchain_ollama import ChatOllama

# Initialize the Ollama model for the agent
llm = ChatOllama(
    model="gemma3:270m",
    temperature=0
)

In [14]:
from langchain import hub
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import render_text_description
from langchain.agents import create_react_agent, AgentExecutor

# Pull the ReAct-style agent prompt from the LangChain Hub
# This prompt uses the ReAct (Reasoning and Acting) framework
prompt = hub.pull("hwchase17/react-json")

# You can inspect the prompt to see its structure
# print(prompt.pretty_print())

# The prompt needs to know what tools are available, which you can provide
# This prepares the tool descriptions for the prompt
prompt = prompt.partial(
    tools=render_text_description(tools),
    tool_names=", ".join([t.name for t in tools]),
)

In [15]:
# Create the core agent logic
agent = create_react_agent(llm, tools, prompt)

# Create the agent executor to run the agent
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [16]:
response = agent_executor.invoke({
    "input": "What is the current date and time?"
})

print(response)



> Entering new AgentExecutor chain...


ValueError: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Could not parse LLM output: ````
$JSON_BLOB
```
`
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 

In [3]:
from langchain import hub
from langchain.agents import create_json_agent, AgentExecutor
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
import datetime

# Step 1: Define the Tool (same as before)
@tool
def get_current_datetime(format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """
    Returns the current date and time, formatted according to the provided Python strftime format string.
    Use this tool whenever the user asks for the current date, time, or both.
    Example format strings: '%Y-%m-%d' for date, '%H:%M:%S' for time.
    If no format is specified, defaults to '%Y-%m-%d %H:%M:%S'.
    """
    try:
        return datetime.datetime.now().strftime(format)
    except Exception as e:
        return f"Error formatting date/time: {e}"

tools = [get_current_datetime]

# Step 2: Set Up the LLM (same as before)
llm = ChatOllama(
    model="gemma3:270m",
    temperature=0
)

# Step 3: Create the JSON-based Agent
# Use create_json_agent, which is better at handling the native JSON output of many models
agent = create_json_agent(
    llm,
    tools=tools,
    verbose=True, # Keep verbose for debugging
)

# Step 4: Invoke the Agent
response = agent.invoke({
    "input": "What is the current date and time?"
})

print(response)

TypeError: create_json_agent() missing 1 required positional argument: 'toolkit'

In [6]:
from langchain_core.tools import tool
import datetime

# The function name is 'get_current_date_and_time'
@tool
def get_current_date_and_time(format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """
    Returns the current date and time, formatted according to the provided Python strftime format string.
    Use this tool whenever the user asks for the current date, time, or both.
    Example format strings: '%Y-%m-%d' for date, '%H:%M:%S' for time.
    If no format is specified, defaults to '%Y-%m-%d %H:%M:%S'.
    """
    try:
        return datetime.datetime.now().strftime(format)
    except Exception as e:
        return f"Error formatting date/time: {e}"

# Make sure the variable you pass to the tools list is the correct one
tools = [get_current_date_and_time]

from langchain_ollama import ChatOllama
from langchain.agents import create_json_chat_agent, AgentExecutor
from langchain import hub

llm = ChatOllama(
    model="gemma3:270m",
    temperature=0
)

prompt = hub.pull("hwchase17/react-chat-json")

agent = create_json_chat_agent(
    llm,
    tools,
    prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

response = agent_executor.invoke({
    "input": "What is the current date and time?"
})

print(response)



> Entering new AgentExecutor chain...
```json
{
  "action": "Get current date and time",
  "action_input": "2024-10-27 10:30:00"
}
```Get current date and time is not a valid tool, try one of [get_current_date_and_time].```json
{
  "action": "Get current date and time",
  "action_input": "2024-10-27 10:30:00"
}
```Get current date and time is not a valid tool, try one of [get_current_date_and_time].```json
{
  "action": "Get current date and time",
  "action_input": "2024-10-27 10:30:00"
}
```Get current date and time is not a valid tool, try one of [get_current_date_and_time].```json
{
  "action": "Get current date and time",
  "action_input": "2024-10-27 10:30:00"
}
```Get current date and time is not a valid tool, try one of [get_current_date_and_time].```json
{
  "action": "Get current date and time",
  "action_input": "2024-10-27 10:30:00"
}
```Get current date and time is not a valid tool, try one of [get_current_date_and_time].```json
{
  "action": "Get current date and time",

KeyboardInterrupt: 

In [7]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
import datetime

@tool
def get_current_date_and_time(format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """Returns the current date and time, formatted according to the provided Python strftime format string."""
    try:
        return datetime.datetime.now().strftime(format)
    except Exception as e:
        return f"Error formatting date/time: {e}"

tools = [get_current_date_and_time]

llm = ChatOllama(
    model="gemma3:270m",
    temperature=0
)

# The prompt for a tool-calling agent is simpler. It's focused on instructing the agent.
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. You have access to the following tools: {tools}"),
    ("user", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# Create the agent
agent = create_tool_calling_agent(llm, tools, prompt)

# Create the executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Invoke the agent
response = agent_executor.invoke({"input": "What is the current date and time?"})
print(response)



> Entering new AgentExecutor chain...


KeyError: "Input to ChatPromptTemplate is missing variables {'tools'}.  Expected: ['input', 'tools'] Received: ['input', 'intermediate_steps', 'agent_scratchpad']\nNote: if you intended {tools} to be part of the string and not a variable, please escape it with double curly braces like: '{{tools}}'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT "

In [12]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

model = OllamaLLM(model="gemma3:270m",temperature=0.0)

template = """
You are an expert in answering question about a pizza restuarant

Here are some relevent reviewa: {reviews}

Here are some question to answer: {question}
"""

# Use PromptTemplate (already imported in another cell) to avoid formatting errors
# and avoid overwriting the existing `prompt` variable used elsewhere.
text_prompt = ChatPromptTemplate.from_template(template)

chain = text_prompt | model

result = chain.invoke({'reviews': [], 'question': 'what is the best pizza place in town?'})
print(result)

Okay, I'm ready to answer your questions about pizza restaurants.



In [13]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

model = OllamaLLM(model="gemma3:270m",temperature=0.0)

template = """
You are an expert in answering question about a pizza restuarant

Here are some relevent reviewa: {reviews}

Here are some question to answer: {question}
"""

# Use PromptTemplate (already imported in another cell) to avoid formatting errors
# and avoid overwriting the existing `prompt` variable used elsewhere.
text_prompt = ChatPromptTemplate.from_template(template)

chain = text_prompt | model


while True:
    print('\n\n-------------------------')
    question = input("Ask your question (q to quite) : ")
    print('\n\n-------------------------')
    if question == 'q':
        break
    result = chain.invoke({'reviews': [], 'question': question})
    print(result)



-------------------------


-------------------------
Okay, I'm ready. Let's begin!



-------------------------


-------------------------
Okay, I'm ready. Please provide the review you'd like me to analyze. I will do my best to provide a comprehensive and insightful response.



-------------------------


-------------------------


In [15]:
import datetime
datetime.datetime.now()

datetime.datetime(2025, 10, 8, 19, 20, 38, 465042)

In [18]:
import datetime
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain.tools import tool

# 1. Use ChatOllama from chat_models, not OllamaLLM from llms
from langchain_ollama.chat_models import ChatOllama

# Note: For tool calling, a larger model is often more reliable.
# gemma3:270m is very small and may struggle. Consider gemma2:9b or llama3:8b if you face issues.
model = ChatOllama(model="gemma3:4b", temperature=0.0)

# Define the format string for strftime
format_string = "%Y-%m-%d %H:%M:%S"

@tool
def get_current_date_and_time() -> str:
    """
    Returns the current date and time.
    Use this tool whenever the user asks for the current date, time, or both.
    """
    # Use the defined format string
    return datetime.datetime.now().strftime(format_string)

tools = [get_current_date_and_time]

# The prompt is correct and doesn't need changes
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI assistant. You have access to the following tools:"),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

# This will now work because 'model' (an instance of ChatOllama) has the .bind_tools() method
agent = create_tool_calling_agent(
    model,
    tools,
    prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True # Set to True to see the agent's thought process
)

# 2. You need to invoke the agent_executor, not a variable named 'chain'
result = agent_executor.invoke({"input": "What is the current date and time?"})
print("\n--- Final Answer ---")
print(result['output'])



> Entering new AgentExecutor chain...


ResponseError: registry.ollama.ai/library/gemma3:4b does not support tools (status code: 400)